# Testing for using VCF that contains SNPs only

In [1]:
# imports 
from cyvcf2 import VCF
import numpy as np
import pandas as pd

In [2]:
# Check the kernel executable
# This is useful to ensure the script is run in the correct Python environment
import sys
print(sys.executable)

/home/tbuser/miniconda3/envs/tbprofiler/bin/python


Add here an explanation of the dataframe using a table

In [ ]:
# Load VCF as plain text (small files)
vcf_file = "/mnt/c/Users/Lenovo/Downloads/vcf/ERR4799720_snps_only.vcf"
	
# Store rows
records = []

with open(vcf_file) as f:
    for line in f:
        if line.startswith("#"):
            continue  # skip header
        parts = line.strip().split("\t")
        chrom = parts[0]
        pos = parts[1]
        ref = parts[3]
        alt = parts[4]
        info = parts[7]
        format_fields = parts[8].split(":")
        sample_fields = parts[9].split(":")
        
        # Example: Extract GT and DP
        format_dict = dict(zip(format_fields, sample_fields))
        gt = format_dict.get("GT", ".")
        dp = format_dict.get("DP", ".")

        # Also parse allele frequency from INFO
        info_dict = dict(item.split("=") for item in info.split(";") if "=" in item)
        af = info_dict.get("AF", ".")

        records.append({
            "CHROM": chrom,
            "POS": int(pos),
            "REF": ref,
            "ALT": alt,
            "GT": gt,
            "DP": dp,
            "AF": af
        })

df = pd.DataFrame(records)
print(df)


         CHROM      POS REF ALT   GT   DP   AF
0   Chromosome     7362   G   C  1/1  133    1
1   Chromosome     7585   G   C  1/1  121    1
2   Chromosome     9304   G   A  1/1  101    1
3   Chromosome   491742   T   C  1/1   87    1
4   Chromosome   657081   C   T  1/1  122    1
5   Chromosome   732036   A   G  1/1  123    1
6   Chromosome   759746   C   T  1/1  105    1
7   Chromosome   762434   T   G  1/1  104    1
8   Chromosome   763031   T   C  1/1  138    1
9   Chromosome   775639   T   C  1/1  116    1
10  Chromosome   776100   G   A  1/1  116    1
11  Chromosome   781395   T   C  1/1  120    1
12  Chromosome  1253127   C   T  0/0   54    0
13  Chromosome  1253130   C   T  0/0   53    0
14  Chromosome  1253131   C   G  0/0   53    0
15  Chromosome  1253135   A   C  0/1   52  0.5
16  Chromosome  1253136   C   G  0/0   50    0
17  Chromosome  1254562   A   G  1/1  127    1
18  Chromosome  1406374   G   C  0/0  110    0
19  Chromosome  1406388   C   A  0/0  116    0
20  Chromosom

In [ ]:
def gt_to_num(gt):
    if gt == "0/0":
        return 0
    elif gt == "0/1" or gt == "1/0":
        return 1
    elif gt == "1/1":
        return 2
    else:
        return None

df["GT_numeric"] = df["GT"].apply(gt_to_num)
df["GT_numeric"]